# Colab Worker — Corpo e Mente

**Modelos cargados:**
- `Qwen3-VL-Embedding-2B` — embeddings multimodales (2048 dims)
- `YOLO26x-pose` — 17 keypoints COCO (x, y) por persona
- `YOLO26x-depth` — profundidad métrica Z en metros reales

**Fusión 3D (RF-03):** cada keypoint se enriquece con `z = depth_map[y_d, x_d]`
en metros absolutos, produciendo coordenadas `(x, y, z)` en $\mathbb{R}^3$.

**Runtime recomendado:** T4 GPU (mínimo) · A100 (ideal para x-large)

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio \
    sentence-transformers langchain-text-splitters \
    opencv-python-headless pillow numpy \
    ultralytics psutil

In [ ]:
# Celda 2: Diagnóstico de hardware + configuración de ngrok
import os
import cv2
import time
import subprocess
import sys
import numpy as np
import requests
import psutil
import torch
from PIL import Image
from pyngrok import ngrok

# --- Diagnóstico de GPU y RAM ---
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🖥️  GPU: {gpu_name}")
    print(f"💾 VRAM total: {vram_total_gb:.2f} GB")
else:
    print("⚠️  No se detectó GPU. Los modelos correrán en CPU (muy lento).")

ram_total_gb = psutil.virtual_memory().total / 1e9
print(f"🧠 RAM total: {ram_total_gb:.2f} GB")

# --- Token de ngrok desde Colab Secrets (obligatorio) ---
from google.colab import userdata
try:
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = None

if not NGROK_TOKEN:
    raise RuntimeError(
        "❌ NGROK_TOKEN no configurado.\n"
        "   Ve a Colab → 🔑 (ícono de llave en el panel lateral) → agrega el secret NGROK_TOKEN.\n"
        "   Obtén uno gratis en https://dashboard.ngrok.com/get-started/your-authtoken"
    )

ngrok.set_auth_token(NGROK_TOKEN)
print(f"✅ Token ngrok configurado: {NGROK_TOKEN[:8]}...")

# Verificar dependencias
import ultralytics
import fastapi, uvicorn
from sentence_transformers import SentenceTransformer
print(f"✅ ultralytics versión: {ultralytics.__version__}")
print("✅ Dependencias verificadas")

In [ ]:
# Celda 3: Carga de los 3 modelos en VRAM
import torch
import logging
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ultralytics import YOLO

logger = logging.getLogger("colab_worker")

# --- 1. Estado inicial de VRAM ---
if torch.cuda.is_available():
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- 2. Cargar Qwen ---
print("\nCargando Qwen3-VL-Embedding-2B...")
qwen_model = SentenceTransformer('Qwen/Qwen3-VL-Embedding-2B')
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
print("✅ Qwen cargado. Dim:", qwen_model.get_sentence_embedding_dimension())

if torch.cuda.is_available():
    print(f"VRAM usada tras Qwen: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras Qwen: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 3. Cargar YOLO26x-pose (con fallback a nano) ---
print("\nCargando YOLO26x-pose...")
try:
    yolo_pose = YOLO('yolo26x-pose.pt')
    print("✅ YOLO26x-pose cargado")
except Exception as e:
    logger.warning(f"No se pudo cargar yolo26x-pose.pt ({e}), cayendo a yolo26n-pose.pt")
    print(f"⚠️  Fallback: cargando yolo26n-pose.pt en lugar de x-large")
    print(f"⚠️  Causa: {e}")
    yolo_pose = YOLO('yolo26n-pose.pt')
    print("✅ YOLO26n-pose cargado (fallback nano)")

# FIX 1: mover a CUDA explícitamente
if torch.cuda.is_available():
    yolo_pose.to('cuda')
    print("✅ YOLO26-pose movido a CUDA")

if torch.cuda.is_available():
    print(f"VRAM usada tras YOLO-pose: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras YOLO-pose: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 4. Cargar YOLO26x-depth (con fallback a nano) ---
print("\nCargando YOLO26x-depth...")
try:
    yolo_depth = YOLO('yolo26x-depth.pt')
    print("✅ YOLO26x-depth cargado")
except Exception as e:
    logger.warning(f"No se pudo cargar yolo26x-depth.pt ({e}), cayendo a yolo26n-depth.pt")
    print(f"⚠️  Fallback: cargando yolo26n-depth.pt en lugar de x-large")
    print(f"⚠️  Causa: {e}")
    yolo_depth = YOLO('yolo26n-depth.pt')
    print("✅ YOLO26n-depth cargado (fallback nano)")

# FIX 1: mover a CUDA explícitamente
if torch.cuda.is_available():
    yolo_depth.to('cuda')
    print("✅ YOLO26-depth movido a CUDA")

if torch.cuda.is_available():
    print(f"VRAM usada tras YOLO-depth: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras YOLO-depth: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 5. Verificar coexistencia ---
print("\nVerificando que Qwen sigue operativo...")
test_emb = qwen_model.encode(["prueba de coexistencia"], normalize_embeddings=True)
print(f"✅ Qwen responde. Dim del vector: {len(test_emb[0])}")

print("\n✅ Coexistencia triple verificada: Qwen + YOLO26-pose + YOLO26-depth")

In [ ]:
# Celda 4: Endpoints FastAPI — /health, /embed_text, /extraer_poses (con fusión 3D)
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import os
import cv2
import numpy as np
import torch

app = FastAPI(title="Corpo e Mente Colab Worker")


@app.get("/health")
async def health():
    return {"status": "ok", "model": "Qwen3-VL-Embedding-2B + YOLO26x-pose + YOLO26x-depth"}


@app.post("/embed_text")
async def embed_text(texto: str = Form(...)):
    chunks = splitter.split_text(texto)
    embeddings = qwen_model.encode(chunks, normalize_embeddings=True)
    return JSONResponse({
        "chunks": chunks,
        "embeddings": embeddings.tolist(),
        "dimension": 2048,
        "chunk_size": 1500,
        "chunk_overlap": 200
    })


@app.post("/extraer_poses")
async def extraer_poses(video: UploadFile = File(...)):
    """
    Extrae keypoints 3D por frame del video mediante fusión YOLO26-pose + YOLO26-depth.

    Retorna por cada frame:
      - keypoints133: lista de 133 floats = (x, y, z) × 17 + padding(82)
      - frame_idx: índice del frame
      - angulos_articulares: {} (se calcula en el backend)

    x, y: coordenadas en píxeles del frame original.
    z: profundidad en METROS reales (YOLO26x-depth, depth_map[y_d, x_d]).
    """
    temp_path = f"/tmp/{video.filename}"
    with open(temp_path, "wb") as f:
        f.write(await video.read())

    frame_idx = 0
    try:
        torch.cuda.empty_cache()

        # FIX 2: device=0 explícito; conf=0.5 en pose; imgsz=768 en depth
        results_pose = yolo_pose.predict(
            source=temp_path, device=0, stream=False, verbose=False, conf=0.5
        )
        results_depth = yolo_depth.predict(
            source=temp_path, device=0, stream=False, verbose=False, imgsz=768
        )

        # FIX 3: guard de alineación — zip trunca silenciosamente si difieren
        if len(results_pose) != len(results_depth):
            raise RuntimeError(
                f"Desalineación de frames entre pose ({len(results_pose)}) "
                f"y depth ({len(results_depth)}). Revisar modelos YOLO."
            )

        # Obtener resolución original del video para escalar coordenadas de depth
        cap = cv2.VideoCapture(temp_path)
        ret, first_frame = cap.read()
        if ret and first_frame is not None:
            orig_h, orig_w = first_frame.shape[:2]
        else:
            orig_h, orig_w = 640, 640
        cap.release()

        esqueletos = []

        # FIX 4: RuntimeError del guard se convierte en 400, no en 500
        try:
            for r_pose, r_depth in zip(results_pose, results_depth):
                if r_pose.keypoints is not None and len(r_pose.keypoints.data) > 0:
                    # Extraer coordenadas 2D (x, y) de la primera persona detectada
                    kpts_2d = r_pose.keypoints.xy.cpu().numpy()
                    person_kpts = kpts_2d[0]  # shape: (17, 2)

                    # Extraer mapa de profundidad métrica
                    depth_map = r_depth.depth.data.cpu().numpy()
                    h_depth, w_depth = depth_map.shape

                    # Factores de escala: resolución de depth != resolución original
                    sx = w_depth / orig_w
                    sy = h_depth / orig_h

                    # Fusión 3D: asignar Z métrico a cada keypoint 2D
                    kpts_3d = []
                    for kp_idx, (x, y) in enumerate(person_kpts):
                        x_d = int(np.clip(x * sx, 0, w_depth - 1))
                        y_d = int(np.clip(y * sy, 0, h_depth - 1))
                        z_metrico = float(depth_map[y_d, x_d])

                        kpts_3d.extend([float(x), float(y), z_metrico])

                    # Padding a 133 dims: (x, y, z) × 17 = 51, padding = 82 zeros
                    if len(kpts_3d) < 133:
                        kpts_3d.extend([0.0] * (133 - len(kpts_3d)))
                    elif len(kpts_3d) > 133:
                        kpts_3d = kpts_3d[:133]

                    esqueletos.append({
                        "frame_idx": frame_idx,
                        "keypoints133": kpts_3d,
                        "angulos_articulares": {}
                    })
                frame_idx += 1
        except RuntimeError as e:
            return JSONResponse(
                {"error": str(e), "total_frames_procesados": frame_idx},
                status_code=400
            )

        torch.cuda.empty_cache()

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    if not esqueletos:
        return JSONResponse(
            {"error": "No se detectaron poses en el video", "total_frames_procesados": frame_idx},
            status_code=400
        )

    return JSONResponse({
        "total_frames_procesados": frame_idx,
        "total_esqueletos": len(esqueletos),
        "esqueletos": esqueletos
    })

In [ ]:
# Celda 5: Levantar uvicorn + túnel ngrok
import threading

# 1. Levantar uvicorn en un hilo, forzando IPv4
def _run_uvicorn():
    import asyncio
    from uvicorn.config import Config
    from uvicorn.server import Server

    # Create a uvicorn Config object
    config = Config(app, host="127.0.0.1", port=8000, log_level="warning")
    # Instantiate the server
    server = Server(config=config)
    # Run the server's serve method using the patched asyncio.run,
    # which avoids uvicorn's internal call to asyncio_run with loop_factory.
    asyncio.run(server.serve())

threading.Thread(target=_run_uvicorn, daemon=True).start()

# 2. Esperar a que uvicorn escuche localmente
print("⏳ Esperando a que uvicorn levante...")
uvicorn_ok = False
for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print(f"✅ uvicorn listo en {i+1}s: {r.json()}")
            uvicorn_ok = True
            break
    except Exception:
        pass
    time.sleep(1)

if not uvicorn_ok:
    raise RuntimeError("❌ uvicorn no respondió en 30s. Revisa errores de importación.")

# 3. Conectar ngrok apuntando a IPv4 explícita
ngrok.kill()  # limpiar túneles previos
ngrok_tunnel = ngrok.connect("http://127.0.0.1:8000", bind_tls=True)

print("=" * 60)
print(f"COLAB_TUNNEL_URL = {ngrok_tunnel.public_url}")
print("Copia esta URL en tu backend/.env como COLAB_TUNNEL_URL")
print("=" * 60)

# 4. Probar el túnel público
time.sleep(2)
try:
    r = requests.get(f"{ngrok_tunnel.public_url}/health", timeout=15)
    print(f"🌐 Túnel público → {r.status_code}: {r.text}")
except Exception as e:
    print(f"⚠️ No se pudo probar el túnel público: {e}")
    print("Pero uvicorn está corriendo localmente. Prueba desde tu navegador.")

In [ ]:
from google.colab import runtime
runtime.unassign()
